In [13]:
import pandas as pd
from pathlib import Path   
import numpy as np

In [14]:
result_path = Path("../results/downstream_task")
methods_list = ["uniform","psa", "kmm", "mrs-forest", "fw-mrs-temperature",  "fw-mrs-temperature-svm",   
"soft-mrs-linear", "soft-mrs-exponential"]
# bias_types = ["less_negative_class", "less_positive_class", "mean_difference"]
bias_types = ["less_positive_class"]
metrics = ["AUROC", "AUPRC"]
# less_bias_strengths = ["0.1", "0.2", "0.3"]
less_bias_strengths = ["0.1"]
mean_bias_strengthts = ["0.8", "0.9"]
datasets = ["folktables_employment", "folktables_income", "hr_analytics", "breast_cancer", "loan_prediction"]

In [15]:
aurocs = []
auprcs = []
dict_list = []
for dataset in datasets:
    for bias_type in bias_types:
        for method in methods_list:
            if bias_type == "mean_difference":
                bias_strengths = mean_bias_strengthts
            else : 
                bias_strengths = less_bias_strengths
            for bias_strength in bias_strengths:
                json_file = result_path / dataset / bias_type /  bias_strength/ method / "classification_results.json"
                result_file = pd.read_json(str(json_file))
                dict_list.append(
                    {
                        "Method": method, "Data Set": dataset, 
                        "AUROC Mean": result_file["random forest auroc"]["mean"], 
                        "AUROC Std": result_file["random forest auroc"]["sd"], 
                        "AUPRC Mean": result_file["random forest auprc"]["mean"], 
                        "AUPRC Std": result_file["random forest auprc"]["sd"], 
                        "Bias Type": bias_type, "Bias Strength": bias_strength,
                        "Dropped Samples Mean": result_file["dropped_samples"]["mean"],
                        "Dropped Samples Std": result_file["dropped_samples"]["std"]
                    }
                                )
result_df = pd.DataFrame(data=dict_list)

In [16]:
result_df = result_df.replace({"uniform": "Uniform", "kmm": "KMM", "psa": "PSA", "mrs-forest": "MRS",
                               "soft-mrs-linear": "Soft-MRS-Linear", "soft-mrs-exponential": "Soft-MRS-Exponential",
                               "fw-mrs-temperature-svm": "FW-MRS-SVM", "fw-mrs-temperature": "FW-MRS-RF"})
result_df


,Method,Data Set,AUROC Mean,AUROC Std,AUPRC Mean,AUPRC Std,Bias Type,Bias Strength,Dropped Samples Mean,Dropped Samples Std
0,Uniform,folktables_employment,0.870831,0.010040,0.827021,0.016671,less_positive_class,0.1,0.00,0.000000
1,PSA,folktables_employment,0.867422,0.010838,0.823830,0.016861,less_positive_class,0.1,0.04,0.280000
2,KMM,folktables_employment,0.857216,0.012799,0.809631,0.020233,less_positive_class,0.1,0.00,0.000000
3,MRS,folktables_employment,0.868837,0.010245,0.825683,0.016146,less_positive_class,0.1,232.80,41.667253
4,FW-MRS-RF,folktables_employment,0.860363,0.011496,0.815249,0.017594,less_positive_class,0.1,169.20,47.993333
5,FW-MRS-SVM,folktables_employment,0.833207,0.015159,0.777130,0.024068,less_positive_class,0.1,232.50,31.036269
6,Soft-MRS-Linear,folktables_employment,0.862929,0.012959,0.819353,0.020234,less_positive_class,0.1,0.00,0.000000
7,Soft-MRS-Exponential,folktables_employment,0.863699,0.011914,0.819417,0.019061,less_positive_class,0.1,0.00,0.000000
8,Uniform,folktables_income,0.838550,0.012914,0.788787,0.018468,less_positive_class,0.1,0.00,0.000000
9,PSA,folktables_income,0.831383,0.012381,0.781471,0.018295,less_positive_class,0.1,0.10,0.412311


In [17]:
for bias_type in bias_types:
    if bias_type == "mean_difference":
        bias_strengths = mean_bias_strengthts
    else : 
        bias_strengths = less_bias_strengths
    for bias_strength in bias_strengths:
        print(f"{bias_type}, {bias_strength}")
        for method in result_df["Method"].unique():
            mean_auroc_values = []
            std_auroc_values = []
            for dataset in datasets:
                mean_auroc = result_df.loc[(result_df["Method"]==method) & (result_df["Bias Type"]==bias_type) & 
                                                    (result_df["Bias Strength"]==bias_strength) & 
                                                    (result_df["Data Set"]==dataset)]["AUROC Mean"].iloc[0]
                mean_auroc_values.append(np.round(mean_auroc, 3))

                std_auroc = result_df.loc[(result_df["Method"]==method) & (result_df["Bias Type"]==bias_type) & 
                                                    (result_df["Bias Strength"]==bias_strength) & 
                                                    (result_df["Data Set"]==dataset)]["AUROC Std"].iloc[0]
                std_auroc_values.append(np.round(std_auroc, 3))

            print(f"\t& {method} \
& ${mean_auroc_values[0]}\\pm{std_auroc_values[0]}$ \
& ${mean_auroc_values[1]}\\pm{std_auroc_values[1]}$ \
& ${mean_auroc_values[2]}\\pm{std_auroc_values[2]}$ \
& ${mean_auroc_values[3]}\\pm{std_auroc_values[3]}$ \
& ${mean_auroc_values[4]}\\pm{std_auroc_values[4]}$ & & \\\\")
        print("\n")

less_positive_class, 0.1
	& Uniform & $0.871\pm0.01$ & $0.839\pm0.013$ & $0.752\pm0.018$ & $0.988\pm0.006$ & $0.672\pm0.075$ & & \\
	& PSA & $0.867\pm0.011$ & $0.831\pm0.012$ & $0.753\pm0.019$ & $0.988\pm0.006$ & $0.648\pm0.089$ & & \\
	& KMM & $0.857\pm0.013$ & $0.819\pm0.015$ & $0.749\pm0.019$ & $0.99\pm0.005$ & $0.62\pm0.084$ & & \\
	& MRS & $0.869\pm0.01$ & $0.838\pm0.012$ & $0.752\pm0.018$ & $0.99\pm0.005$ & $0.649\pm0.08$ & & \\
	& FW-MRS-RF & $0.86\pm0.011$ & $0.833\pm0.013$ & $0.751\pm0.021$ & $0.989\pm0.005$ & $0.626\pm0.071$ & & \\
	& FW-MRS-SVM & $0.833\pm0.015$ & $0.822\pm0.016$ & $0.753\pm0.019$ & $0.985\pm0.008$ & $0.571\pm0.082$ & & \\
	& Soft-MRS-Linear & $0.863\pm0.013$ & $0.825\pm0.013$ & $0.754\pm0.019$ & $0.989\pm0.005$ & $0.622\pm0.088$ & & \\
	& Soft-MRS-Exponential & $0.864\pm0.012$ & $0.825\pm0.012$ & $0.755\pm0.018$ & $0.989\pm0.006$ & $0.622\pm0.095$ & & \\




In [18]:
for bias_type in bias_types:
    if bias_type == "mean_difference":
        bias_strengths = mean_bias_strengthts
    else : 
        bias_strengths = less_bias_strengths
    for bias_strength in bias_strengths:
        print(f"{bias_type}, {bias_strength}")
        for method in result_df["Method"].unique():
            mean_auprc_values = []
            std_auprc_values = []
            for dataset in datasets:
                mean_auprc = result_df.loc[(result_df["Method"]==method) & (result_df["Bias Type"]==bias_type) & 
                                                    (result_df["Bias Strength"]==bias_strength) & 
                                                    (result_df["Data Set"]==dataset)]["AUPRC Mean"].iloc[0]
                mean_auprc_values.append(np.round(mean_auprc, 3))

                std_auprc = result_df.loc[(result_df["Method"]==method) & (result_df["Bias Type"]==bias_type) & 
                                                    (result_df["Bias Strength"]==bias_strength) & 
                                                    (result_df["Data Set"]==dataset)]["AUPRC Std"].iloc[0]
                std_auprc_values.append(np.round(std_auprc, 3))

            print(f"\t& {method} \
& ${mean_auprc_values[0]}\\pm{std_auprc_values[0]}$ \
& ${mean_auprc_values[1]}\\pm{std_auprc_values[1]}$ \
& ${mean_auprc_values[2]}\\pm{std_auprc_values[2]}$ \
& ${mean_auprc_values[3]}\\pm{std_auprc_values[3]}$ \
& ${mean_auprc_values[4]}\\pm{std_auprc_values[4]}$ & & \\\\")
        print("\n")

less_positive_class, 0.1
	& Uniform & $0.827\pm0.017$ & $0.789\pm0.018$ & $0.456\pm0.033$ & $0.994\pm0.003$ & $0.804\pm0.048$ & & \\
	& PSA & $0.824\pm0.017$ & $0.781\pm0.018$ & $0.457\pm0.032$ & $0.994\pm0.003$ & $0.794\pm0.056$ & & \\
	& KMM & $0.81\pm0.02$ & $0.764\pm0.02$ & $0.448\pm0.034$ & $0.995\pm0.003$ & $0.782\pm0.053$ & & \\
	& MRS & $0.826\pm0.016$ & $0.789\pm0.019$ & $0.456\pm0.03$ & $0.995\pm0.002$ & $0.795\pm0.05$ & & \\
	& FW-MRS-RF & $0.815\pm0.018$ & $0.785\pm0.019$ & $0.454\pm0.037$ & $0.995\pm0.003$ & $0.784\pm0.046$ & & \\
	& FW-MRS-SVM & $0.777\pm0.024$ & $0.773\pm0.022$ & $0.46\pm0.033$ & $0.992\pm0.005$ & $0.755\pm0.051$ & & \\
	& Soft-MRS-Linear & $0.819\pm0.02$ & $0.773\pm0.019$ & $0.457\pm0.034$ & $0.995\pm0.003$ & $0.783\pm0.056$ & & \\
	& Soft-MRS-Exponential & $0.819\pm0.019$ & $0.772\pm0.019$ & $0.461\pm0.033$ & $0.995\pm0.003$ & $0.783\pm0.059$ & & \\


